# NyayaVani: Sovereign Indic Legal SLM Fine-Tuning (1-Click Colab T4)
Fine-tunes **Llama-3.2-3B-Instruct** on NyayaVani's Indian Legal Incident Dataset using **QLoRA (4-bit)**.
Generates an ultra-lightweight ~50 MB LoRA adapter that plugs directly into the NyayaVani legal backend.

In [ ]:
# 1. Install dependencies
!pip install -q -U torch transformers peft trl datasets bitsandbytes accelerate

In [ ]:
# 2. Check GPU environment
import torch
print('Active Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected - enable GPU in Runtime > Change runtime type')

In [ ]:
# 3. Upload dataset or download directly
from google.colab import files
print('Upload nyayavani_finetune_dataset.jsonl from your local project scripts/data/ directory:')
uploaded = files.upload()

In [ ]:
# 4. QLoRA 4-bit Model Loading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Train with HuggingFace SFTTrainer
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset('json', data_files='nyayavani_finetune_dataset.jsonl', split='train')

training_args = TrainingArguments(
    output_dir="nyayavani_lora_colab",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    bf16=True,
    save_strategy="epoch"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="input",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args
)

print('Training starting (takes ~35-40 minutes on T4 GPU)...')
trainer.train()

In [ ]:
# 6. Save and Download LoRA Adapter
model.save_pretrained("nyayavani_lora_final")
tokenizer.save_pretrained("nyayavani_lora_final")
!zip -r nyayavani_lora_adapter.zip nyayavani_lora_final/

from google.colab import files
files.download('nyayavani_lora_adapter.zip')
print('Done! Unzip this 50MB folder into your NyayaVani project.')